2 (10 points). In the previous part you did a targeted attack on a network where the
target class t (for the adversarial input) was 0. For this part, repeat the attack for the
target class t = 1. If you were successful, report your results. If unsuccessful, explain
what is happening: does this mean that the FGSM attack is ineffective for the given
network? Play around with the parameters or use other techniques to find a close enough
x′ that fools the classifier. Report your results, including the method you used and the
norm of the difference between your adversarial input x′ and the given input x.

In [57]:
import torch
import torch.nn as nn


# fix seed so that random initialization always performs the same
torch.manual_seed(13)


# create the model N as described in the question
N = nn.Sequential(nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 10, bias=False),
                  nn.ReLU(),
                  nn.Linear(10, 3, bias=False))

# random input
x = torch.rand((1,10)) # the first dimension is the batch size; the following dimensions the actual dimension of the data
x.requires_grad_() # this is required so we can compute the gradient w.r.t x

t = 1 # target class

# The network N classfies x as belonging to class 2
original_class = N(x).argmax(dim=1).item()  # TO LEARN: make sure you understand this expression
print("Original Class: ", original_class)
assert(original_class == 2)

# compute gradient
# note that CrossEntropyLoss() combines the cross-entropy loss and an implicit softmax function
L = nn.CrossEntropyLoss()
loss = L(N(x), torch.tensor([t], dtype=torch.long)) # TO LEARN: make sure you understand this line
print(loss)
loss.backward()

Original Class:  2
tensor(1.1157, grad_fn=<NllLossBackward0>)


## The Original Approach
This is a single step FGSM with epsReal=0.5

**Compute perturbation.**

$$\eta = \epsilon \cdot \operatorname{sign}\big(\nabla_x \, \text{loss}_t(x)\big)$$

**Perturb the input.**

$$x' = x - \eta$$

In [58]:
epsReal = 0.5
eps = epsReal - 1e-7
adv_x = x - eps*torch.sign(x.grad)

In [59]:
new_class = N(adv_x).argmax(dim=1).item()
print("New Class: ", new_class)

New Class:  2


As seen above, the target class t=1 was not achieved.

## Approach 1: Epsilon Sweep

Here we start from a small value for epsilon and increase it gradually until the neighbourhood is large enough to find an adversarial x for the target t.

In [60]:
adv_x = 0
grad = x.grad
epsReal = 0.001  # We will be sweeping epsReal
eps = epsReal - 1e-7 # small constant to offset floating-point erros
print(x, grad)

tensor([[0.6584, 0.8175, 0.1262, 0.0541, 0.2268, 0.8405, 0.5393, 0.9798, 0.8597,
         0.5977]], requires_grad=True) tensor([[ 0.0420, -0.0004, -0.0112,  0.0155,  0.0447, -0.0180,  0.0227,  0.0003,
         -0.0204,  0.0019]])


In [61]:
# Sweep epsReal with ea
N_Sweep = int(1e4)
eps_del = 0.001
for i in range(N_Sweep):
  epsReal += eps_del
  eps = epsReal - 1e-7
  adv_x = x - eps*torch.sign(grad)
  new_class = N(adv_x).argmax(dim=1).item()
  if new_class == t:
    print("epsReal found:", epsReal)
    print("New Class: ", new_class)
    print("adv_x: ", adv_x)
    print("Loss: ", L(N(adv_x), torch.tensor([t], dtype=torch.long)))
    break

epsReal found: 1.1379999999999855
New Class:  1
adv_x:  tensor([[-0.4796,  1.9555,  1.2642, -1.0839, -0.9112,  1.9785, -0.5987, -0.1582,
          1.9977, -0.5403]], grad_fn=<SubBackward0>)
Loss:  tensor(1.0963, grad_fn=<NllLossBackward0>)


In [62]:
new_class = N(adv_x).argmax(dim=1).item()
print("New Class: ", new_class)
assert(new_class == t)
# it is not enough that adv_x is classified as t. We also need to make sure it is 'close' to the original x.
print(torch.norm((x-adv_x),  p=float('inf')).data)
assert(torch.norm((x-adv_x), p=float('inf')) <= epsReal)

New Class:  1
tensor(1.1380)


## Approach 2: Targeted Iterative FGSM

Here, we aim to iteratively update the adv_x with the original epsReal=0.5 and step size parameter alpha=0.25. We take steps until the model output equals the target class t.

In [63]:
# Revert epsilon values to the ones used in Part 1
alpha = 0.25
epsReal = 0.5
eps = epsReal - 1e-7

In [64]:
adv_x = x.clone().detach().requires_grad_()
model_output = N(adv_x).argmax(dim=1).item()
steps = 0
max_steps = int(1e2)

# Iterate until an adversarial x is found for target t.
while(model_output != t and steps < max_steps):
  loss = L(N(adv_x), torch.tensor([t], dtype=torch.long))
  loss.backward()
  grad = adv_x.grad
  adv_x = adv_x - alpha*torch.sign(grad) # FGSM step towards the target

  # Clip the adv_x to -eps to +eps neighbourhood of x.
  adv_x = x + torch.clamp(adv_x - x, -eps, eps)
  adv_x = adv_x.detach().requires_grad_()
  model_output = N(adv_x).argmax(dim=1).item()
  steps += 1

assert(model_output == t)
print("Adversarial X found in step #", steps)
print("adv_x: ", adv_x)
print("New Class: ", model_output)
print(torch.norm((x-adv_x),  p=float('inf')).data)
assert(torch.norm((x-adv_x), p=float('inf')) <= epsReal)

Adversarial X found in step # 18
adv_x:  tensor([[0.1584, 0.8175, 0.6262, 0.3041, 0.2268, 0.8405, 0.0393, 0.4798, 1.3597,
         0.0977]], requires_grad=True)
New Class:  1
tensor(0.5000)
